In [13]:
#Import all necessary libraries
import json
import re
import numpy as np
import pandas as pd
import altair as alt
from collections import Counter

import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download("stopwords")

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/u251664/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

1. As a first step, you must pre-process the documents. In particular, for the text fields (title, 
description) you should:  

● Removing stop words 

● Tokenization 

● Removing punctuation marks 

● Stemming 

● and... anything else you think it's needed (bonus point) 

In [14]:
docs_path = "../../data/fashion_products_dataset.json"

with open(docs_path, "r", encoding="utf-8") as fp:
    data = json.load(fp)

df = pd.DataFrame(data)
print("Total number of products in the corpus: {}".format(len(df)))

Total number of products in the corpus: 28080


In [ ]:
#We have collected this code from Practice 1, doing some minor changes and adding the bonus

stemmer = PorterStemmer()                
stop_words = set(stopwords.words("english"))
extra_stop_words = set() # bonus

def build_terms(line):
    """
    Preprocess the article text (title + body): removing stop words, stemming,
    transforming in lowercase and return the tokens of the text.

    Argument:
    line -- string (text) to be preprocessed

    Returns:
    line - a list of tokens corresponding to the input text after the preprocessing
    """


    if not isinstance(line, str):                      # missing values (None / NaN)
        return []
    line = line.lower()                                # lowercase
    line = re.sub(r"[^a-z0-9\s]", " ", line)           # remove punctuation marks
    line = line.split()                                # Tokenize the text to get a list of terms
    line = [x for x in line if x not in stop_words and x not in extra_stop_words]
    line = [stemmer.stem(word) for word in line]       #  perform stemming
    return line

In [10]:
text = df.loc[0, "description"]
print("Original:\n{}\n".format(text))
print("Terms:\n{}\n".format(build_terms(text)))
print(build_terms("100% Cotton, Men's T-Shirt!!"), build_terms(""), build_terms(None))

Original:
Yorker trackpants made from 100% rich combed cotton giving it a rich look.Designed for Comfort,Skin friendly fabric,itch-free waistband & great for all year round use Proudly made in India

Terms:
['yorker', 'trackpant', 'made', '100', 'rich', 'comb', 'cotton', 'give', 'rich', 'look', 'design', 'comfort', 'skin', 'friendli', 'fabric', 'itch', 'free', 'waistband', 'great', 'year', 'round', 'use', 'proudli', 'made', 'india']

['100', 'cotton', 'men', 'shirt'] [] []


In [16]:
df["title_terms"] = df["title"].apply(build_terms)
df["description_terms"] = df["description"].apply(build_terms)

print("Products with empty title terms: {}".format((df["title_terms"].apply(len) == 0).sum()))
print("Products with empty description terms: {}".format((df["description_terms"].apply(len) == 0).sum()))

Products with empty title terms: 0
Products with empty description terms: 11150


In [17]:
term_counts = Counter(t for terms in df["description_terms"] for t in terms)
print(term_counts.most_common(30))

[('shirt', 18321), ('cotton', 10139), ('wear', 9353), ('comfort', 8836), ('look', 6413), ('casual', 6034), ('fit', 5831), ('made', 5643), ('fabric', 5516), ('print', 4888), ('women', 4424), ('design', 4301), ('sleev', 4221), ('qualiti', 4087), ('men', 3775), ('wash', 3731), ('day', 3617), ('style', 3275), ('make', 3233), ('100', 3210), ('feel', 3160), ('neck', 2991), ('perfect', 2973), ('soft', 2742), ('great', 2740), ('pair', 2734), ('stylish', 2716), ('keep', 2669), ('polo', 2623), ('product', 2597)]


2. Take into account that for future queries, the final output must return (when present) the 
following information for each of the selected documents:  
pid, title, description,  brand,  category, sub_category, product_details, seller, out_of_stock, 
selling_price, discount, actual_price, average_rating, url 

In [18]:
output_fields = ["pid", "title", "description", "brand", "category", "sub_category",
                 "product_details", "seller", "out_of_stock", "selling_price",
                 "discount", "actual_price", "average_rating", "url"]

missing_fields = [f for f in output_fields if f not in df.columns]
print("Missing output fields: {}".format(missing_fields))   # should be []

Missing output fields: []


3. Decide how to handle the fields category, sub_category, brand, product_details, and seller 
during pre-processing. Should they be merged into a single text field, indexed as separate fields 
in the inverted index or any other alternative? Justify your choice, considering how their 
distinctiveness may affect retrieval effectiveness. What are pros and cons of each approach? 

In [20]:
def flatten_product_details(details):
    """
    Convert product_details (list of one-key dicts) into a single text string.

    Argument:
    details -- list of dicts, e.g. [{"Fabric": "Cotton Blend"}, {"Color": "Blue"}]

    Returns:
    string with all keys and values, e.g. "Fabric Cotton Blend Color Blue"
    """
    if not isinstance(details, list):
        return ""
    parts = []
    for item in details:
        if isinstance(item, dict):
            for key, value in item.items():
                parts.append(str(key))
                parts.append(str(value))
    return " ".join(parts)

df["product_details_text"] = df["product_details"].apply(flatten_product_details)

df["details_terms"] = df["product_details_text"].apply(build_terms)
df["brand_terms"] = df["brand"].apply(build_terms)
df["seller_terms"] = df["seller"].apply(build_terms)
df["category_terms"] = (df["category"].fillna("") + " " + df["sub_category"].fillna("")).apply(build_terms)

df[["pid", "product_details_text", "details_terms", "brand_terms", "category_terms"]].head()

,pid,product_details_text,details_terms,brand_terms,category_terms
0,TKPFCZ9EA7H5FYZH,Style Code 1005COMBO2 Closure Elastic Pockets ...,"[style, code, 1005combo2, closur, elast, pocke...",[york],"[cloth, accessori, bottomwear]"
1,TKPFCZ9EJZV2UVRZ,"Style Code 1005BLUE Closure Drawstring, Elasti...","[style, code, 1005blue, closur, drawstr, elast...",[york],"[cloth, accessori, bottomwear]"
2,TKPFCZ9EHFCY5Z4Y,Style Code 1005COMBO4 Closure Elastic Pockets ...,"[style, code, 1005combo4, closur, elast, pocke...",[york],"[cloth, accessori, bottomwear]"
3,TKPFCZ9ESZZ7YWEF,Style Code 1005COMBO3 Closure Elastic Pockets ...,"[style, code, 1005combo3, closur, elast, pocke...",[york],"[cloth, accessori, bottomwear]"
4,TKPFCZ9EVXKBSUD7,"Style Code 1005COMBO1 Closure Drawstring, Elas...","[style, code, 1005combo1, closur, drawstr, ela...",[york],"[cloth, accessori, bottomwear]"


In [19]:
for col in ["category", "sub_category", "brand", "seller"]:
    n_unique = df[col].nunique()
    top_share = df[col].value_counts(normalize=True).iloc[0]
    print("{:<13} distinct values: {:>6} | most common value covers {:.1%}".format(col, n_unique, top_share))

category      distinct values:      4 | most common value covers 96.6%
sub_category  distinct values:     24 | most common value covers 54.5%
brand         distinct values:    325 | most common value covers 7.2%
seller        distinct values:    535 | most common value covers 5.9%


4. Consider the fields out_of_stock, selling_price, discount, actual_price, and average_rating. 
Decide how these should be handled during pre-processing to use in further search. Should 
they be indexed as textual terms?

In [21]:
def parse_number(x):
    """
    Extract the first number from a string such as "2,999" or "69% off".

    Argument:
    x -- string (or number) to parse

    Returns:
    float, or NaN if no number can be extracted
    """
    if isinstance(x, (int, float)):
        return float(x)
    match = re.search(r"\d[\d,]*\.?\d*", str(x))
    return float(match.group().replace(",", "")) if match else np.nan

df["selling_price"] = df["selling_price"].apply(parse_number)
df["actual_price"] = df["actual_price"].apply(parse_number)
df["discount"] = df["discount"].apply(parse_number)
df["average_rating"] = df["average_rating"].apply(parse_number)
df["out_of_stock"] = df["out_of_stock"].astype(bool)

print(df[["selling_price", "actual_price", "discount", "average_rating"]].describe())
print("\nMissing after parsing:\n{}".format(df[["selling_price", "actual_price", "discount", "average_rating"]].isna().sum()))
print("\nout_of_stock:\n{}".format(df["out_of_stock"].value_counts()))
print("\nselling_price <= actual_price in {:.1%} of rows".format((df["selling_price"] <= df["actual_price"]).mean()))

       selling_price  actual_price      discount  average_rating
count   28078.000000  27303.000000  27225.000000    25819.000000
mean      705.635088   1455.528110     50.256896        3.627724
std       549.681489    939.977456     16.887287        0.663429
min        99.000000    150.000000      1.000000        1.000000
25%       390.000000    849.000000     40.000000        3.200000
50%       545.000000   1199.000000     53.000000        3.800000
75%       820.000000   1799.000000     63.000000        4.100000
max      7999.000000  12999.000000     87.000000        5.000000

Missing after parsing:
selling_price        2
actual_price       777
discount           855
average_rating    2261
dtype: int64

out_of_stock:
out_of_stock
False    26436
True      1644
Name: count, dtype: int64

selling_price <= actual_price in 97.2% of rows


In [24]:
LABELS_PATH = "../../data/validation_labels.csv"
labels = pd.read_csv(LABELS_PATH)

print("Labels' pids not found in df: {}".format((~labels["pid"].isin(df["pid"])).sum()))   # should be 0

queries = {1: "women full sleeve sweatshirt cotton", 2: "men slim jeans blue"}
fields = ["title_terms", "description_terms", "details_terms", "brand_terms", "category_terms"]

relevant = labels[labels["labels"] == 1].merge(df, on="pid", how="left", suffixes=("_lab", ""))

for qid, q in queries.items():
    q_terms = set(build_terms(q))
    rel = relevant[relevant["query_id"] == qid]
    print("\nQuery {}: {} -> {}  ({} relevant products)".format(qid, q, sorted(q_terms), len(rel)))
    for f in fields:
        share = rel[f].apply(lambda terms: len(q_terms & set(terms)) / len(q_terms)).mean()
        print("  {:<18} avg share of query terms present: {:.0%}".format(f, share))

Labels' pids not found in df: 0

Query 1: women full sleeve sweatshirt cotton -> ['cotton', 'full', 'sleev', 'sweatshirt', 'women']  (13 relevant products)
  title_terms        avg share of query terms present: 80%
  description_terms  avg share of query terms present: 58%
  details_terms      avg share of query terms present: 80%
  brand_terms        avg share of query terms present: 0%
  category_terms     avg share of query terms present: 0%

Query 2: men slim jeans blue -> ['blue', 'jean', 'men', 'slim']  (10 relevant products)
  title_terms        avg share of query terms present: 80%
  description_terms  avg share of query terms present: 35%
  details_terms      avg share of query terms present: 65%
  brand_terms        avg share of query terms present: 0%
  category_terms     avg share of query terms present: 0%
